# 01 - Perfilado inicial
Dataset crudo: INEGI EDR 2023 (DEFUN23.dbf), 74 variables, ~formato DBF.
Filtro objetivo: `Tipo_defun == 3` (Suicidio / Lesion autoinfligida).
Perfilado con pandas puro (sin ydata-profiling, por compatibilidad con Python 3.14+).

In [ ]:
import pandas as pd
import sys
sys.path.append('../src')
from cleaning_utils import load_dbf, normalize_columns, null_summary, dtype_summary, duplicate_report, profiling_report


## 1. Cargar catalogos (tablas pequenas)
Necesarios para traducir codigos a etiquetas (entidad, municipio, causa CIE-10, etc.)

In [ ]:
cat_geo = load_dbf('../data/raw/CATEMLDE23.dbf')      # Cve_ent, Cve_mun, Cve_loc, Nom_loc
cat_causa = load_dbf('../data/raw/CATMINDE.dbf')       # Cve, Descrip (CIE-10 detallado)
cat_listamex = load_dbf('../data/raw/LISTAMEX.dbf')    # Cve, Descrip (lista mexicana)
cat_capgpo = load_dbf('../data/raw/CAPGPO.dbf')        # Cap, Gpo, Descrip
cat_parentesco = load_dbf('../data/raw/PARENTESCO.dbf')

print('Catalogo geografico:', cat_geo.shape)
print('Catalogo causa (CIE-10):', cat_causa.shape)
print('Catalogo lista mexicana:', cat_listamex.shape)


## 2. Cargar tabla principal DEFUN23.dbf
131 MB, 74 columnas. Puede tardar 1-2 min en cargar.

In [ ]:
df_raw = load_dbf('../data/raw/DEFUN23.dbf')
df_raw = normalize_columns(df_raw)  # TIPO_DEFUN -> Tipo_defun, etc.
print(f'Total de defunciones registradas 2023: {len(df_raw):,}')
df_raw.head()


## 3. Perfilado general (antes de filtrar)

In [ ]:
profiling_report(df_raw)


## 4. Explorar variable clave: Tipo_defun
1=Accidente, 2=Homicidio, 3=Suicidio, 4=Enfermedad, 5=Intervencion legal, 9=Se ignora

In [ ]:
df_raw['Tipo_defun'].value_counts(dropna=False).sort_index()


## 5. Filtrar universo de suicidio
Regla: Tipo_defun == 3. Verificar contra cifra oficial INEGI (~9,085 casos en 2023, 10.8% de 84,118 causas externas).

In [ ]:
df_suicidio = df_raw[df_raw['Tipo_defun'] == 3].copy()
print(f'Registros de suicidio filtrados: {len(df_suicidio):,}')
print('Cifra oficial INEGI (nota tecnica EDR 2023): ~9,085 casos (10.8% de 84,118 causas externas)')


## 6. Perfilado del subconjunto de suicidio

In [ ]:
profiling_report(df_suicidio)


## 7. Exploracion de catalogos relevantes en el subconjunto
Sexo, edad, entidad, mes de ocurrencia, escolaridad, sitio de ocurrencia.

In [ ]:
df_suicidio['Sexo'].value_counts(dropna=False)


In [ ]:
df_suicidio['Ent_ocurr'].value_counts(dropna=False).head(15)


## 8. Hallazgos del perfilado
_Documentar aqui los problemas encontrados (nulos, codigos 'no especificado', inconsistencias) y trasladarlos a docs/methodology.md y docs/quality_report.md_